###  MFCC-Chroma Fusion using a Dual-Channel CNN-LSTM Preprocessing Pipeline
#### Pioneer Edition — Integrates ICBHI 2017 + Hinga (Localized) Datasets

##### Import Statements

In [ ]:
import numpy as np
import pandas as pd
import os
import librosa as lb
import soundfile as sf
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm import tqdm
import shutil
import matplotlib.pyplot as plt
import zipfile
from collections import defaultdict
import random

from scipy.signal import butter, filtfilt
import librosa as lb
import numpy as np

### 1. Initial Configuration 

##### Defined Parameters

In [ ]:
# Target size per class (train only)
TARGET_COUNT = 500                      
SEG_LEN = 3
SR = 4000

##### Paths

In [ ]:
# Pioneer repo structure
# pioneer/
#   icbhi/audio_and_txt_files/
#   icbhi/patient_diagnosis.csv
#   hinga/audioFiles/
#   hinga/audio-disease_df.csv

PIONEER_ROOT = "./pioneer"  # <-- adjust if your pioneer repo is elsewhere

ICBHI_DIR = os.path.join(PIONEER_ROOT, "icbhi")
HINGA_DIR = os.path.join(PIONEER_ROOT, "hinga")

ROOT_DIR = "./"

#Save Directories
SEGMENTS_DIR = "outputs/segmentation_outputs/" 
TRAIN_SEGMENTS_AUGMENTED_DIR = "outputs/augmentation_outputs/train_segments" 
TEST_SEGMENTS_AUGMENTED_DIR = "outputs/augmentation_outputs/test_segments" 
VAL_SEGMENTS_AUGMENTED_DIR = "outputs/augmentation_outputs/val_segments" 


In [ ]:
shutil.rmtree(os.path.join(ROOT_DIR,"outputs"), ignore_errors=True)
shutil.rmtree("./../dataset", ignore_errors=True)

##### Filters and Normalization

The function `load_audio` essentially loads the different audio clips following a 4000Hz sampling rate and applies butterworth filtering prior to returning the audio file. This function will be used to load the **unsegmented** audio files. 

In [ ]:
def butterworth_filter(y, sr, lowcut=50, highcut=1800, order=4):
    nyquist = 0.5 * sr

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, y)

The function `rms_normalize` will be applied to **each segment** aftey they are created and before they are exported to the save directories defined.

In [ ]:
def rms_normalize(y, eps=1e-9, min_rms=1e-3):
    rms = np.sqrt(np.mean(y**2) + eps)
    if rms < min_rms:
        return y  # avoid boosting silence
    return y / rms

##### Augmentation

In [ ]:
def augment_noise(y, sr=SR):
    return y + np.random.normal(0, 0.005, len(y))

def augment_pitch(y, sr=SR):
    steps = np.random.uniform(-2, 2)
    return lb.effects.pitch_shift(y=y, sr=sr, n_steps=steps)

def augment_speed(y, sr=SR):
    rate = np.random.uniform(0.95, 1.05)
    return lb.effects.time_stretch(y=y, rate=rate)

def augment_gain(y, sr=SR):
    return y * np.random.uniform(0.8, 1.2)



# Python list containing all of the augmentation methods
AUGS = [augment_noise, augment_pitch, augment_speed, augment_gain]

### 2. Creating the Reference Data Frame

Two separate metadata readers — one for ICBHI, one for Hinga — then merged into a single combined DataFrame.

**PID Namespacing:** ICBHI PIDs are kept as integers. Hinga PIDs are prefixed with `hinga_` (e.g., `hinga_0`, `hinga_12`) to prevent collisions.

A `source` column and `audio_dir` column track dataset origin and audio file location per recording.


In [ ]:
# Extracts relevant information from the file name
def get_filename_info(file):
    return file.split("_")

In [ ]:
# Extracts relevant information from the file name
def get_filename_info(file):
    return file.split("_")


def read_icbhi_metadata(data_dir):
    """Reads ICBHI metadata. Returns DataFrame with:
    start, end, crackles, wheezes, filename, pid, disease, source, audio_dir
    """
    audio_dir = os.path.join(data_dir, "audio_and_txt_files")
    diagnosis_path = os.path.join(data_dir, "patient_diagnosis.csv")

    patient_df = pd.read_csv(diagnosis_path, names=["pid", "disease"])
    patient_df.pid = patient_df.pid.astype(int)

    txt_files = sorted([f for f in os.listdir(audio_dir) if f.endswith(".txt")])

    rows = []
    for txt in txt_files:
        file = txt.replace(".txt", "")
        metadata = pd.read_csv(
            os.path.join(audio_dir, txt),
            sep="\t",
            names=["start", "end", "crackles", "wheezes"]
        )
        pid = int(get_filename_info(file)[0])
        metadata["filename"] = file
        metadata["pid"] = pid
        rows.append(metadata)

    df = pd.concat(rows)
    df = df.merge(patient_df, on="pid")

    # Drop rare classes (< 3 unique patients) — ICBHI only
    # NOTE: We do NOT drop here for Pioneer because Hinga may fill the gap.
    # Rare class exclusion is done AFTER merging both datasets.

    df["source"] = "icbhi"
    df["audio_dir"] = audio_dir

    print(f"[ICBHI] Patients: {df['pid'].nunique()} | Annotation rows: {len(df)}")
    print(f"[ICBHI] Disease distribution (patients):")
    print(df.groupby("disease")["pid"].nunique().to_string())

    return df.reset_index(drop=True)


def read_hinga_metadata(data_dir):
    audio_dir = os.path.join(data_dir, "audioFiles")
    diagnosis_path = os.path.join(data_dir, "audio-disease_df.csv")

    patient_df = pd.read_csv(diagnosis_path, dtype={"pid": str})  # keep as string e.g. "000"

    # Build a lookup of all wav files in audioFiles indexed by PID prefix
    all_wavs = [f for f in os.listdir(audio_dir) if f.endswith(".wav")]
    pid_to_file = {}
    for wav in all_wavs:
        pid_prefix = wav.split("_")[0]  # e.g. "010" from "010_Healthy_Littmann.wav"
        pid_to_file[pid_prefix] = wav.replace(".wav", "")  # store stem only

    rows = []
    for _, row in patient_df.iterrows():
        original_pid = row["pid"]   # e.g. "010"
        disease = row["disease"]
        namespaced_pid = f"hinga_{original_pid}"

        if original_pid not in pid_to_file:
            print(f"[Hinga] WARNING: no audio file found for PID: {original_pid}")
            continue

        filename_stem = pid_to_file[original_pid]  # e.g. "010_Healthy_Littmann"

        rows.append({
            "start":    None,
            "end":      None,
            "crackles": None,
            "wheezes":  None,
            "filename":  filename_stem,
            "pid":       namespaced_pid,
            "disease":   disease,
            "source":    "hinga",
            "audio_dir": audio_dir,
        })

    df = pd.DataFrame(rows)
    print(f"[Hinga] Patients: {df['pid'].nunique()} | Recordings: {len(df)}")
    print(f"[Hinga] Disease distribution (patients):")
    print(df.groupby("disease")["pid"].nunique().to_string())
    return df.reset_index(drop=True)

# ── Read both datasets ──────────────────────────────────────────
df_icbhi = read_icbhi_metadata(ICBHI_DIR)
df_hinga = read_hinga_metadata(HINGA_DIR)

# ── Merge into combined DataFrame ──────────────────────────────
df = pd.concat([df_icbhi, df_hinga], ignore_index=True)

# ── Drop rare classes AFTER merging (< 3 unique patients) ─────
counts = df.groupby("disease")["pid"].nunique()
rare_classes = counts[counts < 3].index.tolist()
if rare_classes:
    print(f"\nDropping rare classes with < 3 patients: {rare_classes}")
    df = df[~df["disease"].isin(rare_classes)].copy()
else:
    print("\nNo rare classes to drop.")

print(f"\n[PIONEER] Combined dataset:")
print(f"  Total annotation rows: {len(df)}")
print(f"  Total unique patients: {df['pid'].nunique()}")
print(f"  Disease distribution (patients):")
print(df.groupby("disease")["pid"].nunique().to_string())

os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/df_combined.csv", index=False)


### 3. Patient-Level Constrained Stratified Split


This script performs a proper patient-level split for lung sound datasets.

Properties:
- No patient appears in more than one subset
- Guarantees each disease class appears in train, test, unseen (if >= 3 patients)
- 90% development / 10% unseen
- 80/20 train/test inside development
- Reproducible with random seed

Input CSV format (one row per segment or file):
pid, disease, filepath,...

Output:
train_pids.txt
test_pids.txt
val_pids.txt


In [ ]:
# CONFIGURATION
UNSEEN_RATIO = 0.10             # 10%
TEST_RATIO_WITHIN_DEV = 0.20    # 20% of development


def constrained_patient_split(pid_df, seed=42):
    rng = np.random.default_rng(seed)

    counts = pid_df.groupby("disease")["pid"].nunique()
    valid_classes = counts.index

    pid_df = pid_df[pid_df["disease"].isin(valid_classes)].copy()
    print("Classes kept:", list(valid_classes))
    print("Classes removed:", list(counts[counts < 3].index))

    # Group the different PIDs by class
    class_groups = {
        cls: pid_df[pid_df["disease"] == cls]["pid"].unique()
        for cls in valid_classes
    }

    train_pool = []
    test_pids = set()
    unseen_pids = set()

    # Ensure that there is a minimum 1 PID per class per subset
    for cls, pids in class_groups.items():
        pids = list(pids)
        rng.shuffle(pids)
        unseen_pids.add(pids[0])
        test_pids.add(pids[1])
        train_pool.extend(pids[2:])

    all_pids = pid_df["pid"].unique()
    total = len(all_pids)
    target_unseen = int(round(total * UNSEEN_RATIO))

    rng.shuffle(train_pool)
    while len(unseen_pids) < target_unseen and train_pool:
        unseen_pids.add(train_pool.pop())

    dev_pool = [pid for pid in all_pids if pid not in unseen_pids]
    target_test = int(round(len(dev_pool) * TEST_RATIO_WITHIN_DEV))
    remaining = [pid for pid in dev_pool if pid not in test_pids]
    rng.shuffle(remaining)
    while len(test_pids) < target_test and remaining:
        test_pids.add(remaining.pop())

    train_pids = [pid for pid in dev_pool if pid not in test_pids]

    # NOTE: No sorted(map(int,...)) — PIDs are mixed types (int for ICBHI, str for Hinga)
    train_df = pid_df[pid_df["pid"].isin(train_pids)]
    test_df = pid_df[pid_df["pid"].isin(test_pids)]
    val_df = pid_df[pid_df["pid"].isin(unseen_pids)]

    os.makedirs("outputs/constrained_patient_split_outputs", exist_ok=True)
    train_df.to_csv("outputs/constrained_patient_split_outputs/train_df_after_split.csv")
    test_df.to_csv("outputs/constrained_patient_split_outputs/test_df_after_split.csv")
    val_df.to_csv("outputs/constrained_patient_split_outputs/val_df_after_split.csv")

    return train_df, test_df, val_df


train_strat, test_strat, val_strat = constrained_patient_split(df)

for split_name, split_df in [("TRAIN", train_strat), ("TEST", test_strat), ("VAL", val_strat)]:
    print(f"\n--- {split_name} ---")
    for disease, pids in split_df.groupby("disease")["pid"].unique().items():
        print(f"  {disease}: {list(pids)}")


### 4. Segmentation

**Naming Convention**

Segment files follow the format `{filename}_s-{i}.wav` where `i` is the segment index within the recording, starting from 0. Example: `101_1b1_Al_sc_Meditron_s-0.wav`.

**Segmentation strategy — blind sliding window**

Each recording is loaded fully and chunked into non-overlapping 3-second windows starting at sample 0. The last incomplete chunk is dropped (no zero-padding). The disease label is patient-level, inherited from `patient_diagnosis.csv`, so annotation rows are not consulted.

**Why blind segmentation:** real-world deployment will not have crackle/wheeze annotations, and the downstream model does not consume those event labels. Annotation-guided slicing was removed to keep the training distribution consistent with deployment input.

The function `load_audio` essentially loads the different audio clips following a 4000Hz sampling rate and applies butterworth filtering prior to returning the audio file. This function will be used to load the **unsegmented** audio files. 

In [ ]:
def load_audio(file_path, sr=SR): # Loads the audio file following the global parameter SAMPLING RATE
    y, sr = lb.load(file_path, sr=sr)
    y = y.astype(np.float32)

    # Butterworth bandpass filter
    y = butterworth_filter(
        y,
        sr,
        lowcut=50,     # remove heart sounds & DC
        highcut=1800,  # remove high-frequency noise
        order=4
    )

    return np.clip(y, -1, 1), sr

In [ ]:
def extract_segments(df, save_dir, segment_folder):
    """Source-aware segmentation. Uses df['audio_dir'] to locate each recording."""
    seg_folder = os.path.join(save_dir, segment_folder)
    shutil.rmtree(seg_folder, ignore_errors=True)
    os.makedirs(seg_folder)

    segment_list = []

    # One pass per unique recording
    unique_recordings = df.drop_duplicates(subset=["filename"])[["filename", "pid", "disease", "audio_dir"]]

    for _, row in tqdm(unique_recordings.iterrows(), total=len(unique_recordings)):
        file = row["filename"]
        audio_dir = row["audio_dir"]
        wav_path = os.path.join(audio_dir, file + ".wav")

        if not os.path.exists(wav_path):
            print(f"WARNING: file not found, skipping: {wav_path}")
            continue

        y, sr = load_audio(wav_path)

        seg_samples = SEG_LEN * SR
        n_segments = len(y) // seg_samples

        for i in range(n_segments):
            seg = y[i * seg_samples:(i + 1) * seg_samples]
            seg = rms_normalize(seg)

            seg_name = f"{file}_s-{i}.wav"
            seg_path = os.path.join(seg_folder, seg_name)
            sf.write(seg_path, seg, SR)

            segment_list.append({
                "segment_file": seg_name,
                "pid": row["pid"],
                "disease": row["disease"],
            })

    return pd.DataFrame(segment_list)


# NOTE: No data_dir argument needed — audio_dir is in the DataFrame
train_segments = extract_segments(train_strat, SEGMENTS_DIR, "train_segments")
test_segments = extract_segments(test_strat, SEGMENTS_DIR, "test_segments")
val_segments = extract_segments(val_strat, SEGMENTS_DIR, "val_segments")

os.makedirs("outputs/segmentation_outputs", exist_ok=True)
train_segments.to_csv("outputs/segmentation_outputs/train_segments.csv", index=False)
test_segments.to_csv("outputs/segmentation_outputs/test_segments.csv", index=False)
val_segments.to_csv("outputs/segmentation_outputs/val_segments.csv", index=False)

print("\nTrain segments count:", train_segments["pid"].count())
print("Test segments count: ", test_segments["pid"].count())
print("Val segments count:  ", val_segments["pid"].count())

for split_name, seg_df in [("Train", train_segments), ("Test", test_segments), ("Val", val_segments)]:
    print(f"\n{split_name} segments per disease:")
    print(seg_df["disease"].value_counts().to_string())

train_pids = set(train_segments["pid"].unique())
test_pids = set(test_segments["pid"].unique())
val_pids = set(val_segments["pid"].unique())
print("\nPatient overlap between splits (should all be empty):")
print(f"  Train ∩ Test : {train_pids & test_pids}")
print(f"  Train ∩ Val  : {train_pids & val_pids}")
print(f"  Test  ∩ Val  : {test_pids & val_pids}")


### 5. Undersampling and Augmentation. 

**Training Class Only.**

First, we graph the number of segments per disease class to verify how much we have, how much we need to undersample, and how much we need to augment.

##### Segment Distribution

In [ ]:
# Read segments data
seg_df = pd.read_csv('outputs/segmentation_outputs/train_segments.csv')

# Count segments per disease
disease_counts = seg_df['disease'].value_counts()

# Create the bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(disease_counts.index, disease_counts.values, 
               color=['skyblue', 'lightcoral', 'lightgreen', 'orange', 'purple', 'brown', 'gray'])

# Add value labels on bars
for bar, count in zip(bars, disease_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'{count:,}', ha='center', va='bottom', fontweight='bold')

plt.title('Number of Audio Segments per Disease Class', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Number of Segments', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add some padding to top for labels
plt.ylim(0, disease_counts.max() * 1.1)

plt.tight_layout()
plt.show()

# Print the counts
print("Segment counts per disease:")
print(disease_counts)

##### Function

In [ ]:
def augment_training_set(seg_df, source_dir, save_dir, target_count=TARGET_COUNT, seed = 42):
  rng = random.Random(seed)
  np_rng = np.random.default_rng(seed)

  updated_records = []
  undersampling_records = []
  shutil.rmtree(os.path.join(ROOT_DIR,save_dir), ignore_errors=True)
  os.makedirs(os.path.join(ROOT_DIR, save_dir))

  for disease, group in seg_df.groupby("disease"):
    originals = group["segment_file"].tolist()
    pids = group["pid"].tolist()
    files_with_pid = list(zip(originals, pids))
    
    # This carries out the undersampling
    if len(files_with_pid) > target_count:
      # If the data count exists the target number: 
      #   get a subset of the data that is equal to the target count
      files_with_pid = rng.sample(files_with_pid, target_count) # We randomly get the samples
      for f, pid in files_with_pid:
        undersampling_records.append({"segment_file": f, "pid": pid, "disease": disease})

    # At this point no disease class should have data count > 500  
    count = len(files_with_pid)
    print("The current number of disease", disease, "is equal to", count)
    
    # Copy the original segments to the new augmented folder
    #   We do this by copying all of the files under files_with_pid to the save_dir
    for file, pid in files_with_pid:
      base_file = os.path.basename(file)
      src_path = os.path.join(source_dir, base_file)
      dest_path = os.path.join(save_dir, base_file)
      shutil.copy(src_path,dest_path)
      updated_records.append({
        "segment_file": os.path.join(base_file), 
        "pid": pid, 
        "disease": disease,
      })
#--------------------------------------------------------------------------------------
  # 1212 data points are copied to the train_segments_augmented folder at this point
  # 500 COPD              audio files
  # 257 Healthy           audio files
  # 191 URTI              audio files
  # 114 Pneumonia         audio files
  # 92 Bronchiolitis      audio files
  # 58 Bronchiectasis     audio files
  
  # We now attempt to generate
  # 243 Healthy           audio files
  # 309 URTI              audio files
  # 386 Pneumonia         audio files
  # 408 Bronchiolitis     audio files
  # 442 Bronchiectasis    audio files
#--------------------------------------------------------------------------------------
    # Get count of needed augmented files
    current_count = len(files_with_pid)
    needed = target_count - current_count
    if needed > 0:
      print(f"\tAugmenting the disease class {disease}. generating a total number of {needed} samples.")

    i = 1
    for _ in range(needed):
      f, pid = rng.choice(files_with_pid) # We use a random file from the list
      y, sr = lb.load(os.path.join(source_dir,f), sr=SR)

      f = os.path.basename(f)
      

      aug_fn = np_rng.choice(AUGS) # Choose a random augmentation function
      y_aug = aug_fn(y, sr) # Apply the augmentation

      # Force fixed length — augment_speed changes length by ±5%
      seg_samples = SEG_LEN * SR
      if len(y_aug) > seg_samples:
        y_aug = y_aug[:seg_samples]
      elif len(y_aug) < seg_samples:
        y_aug = np.pad(y_aug, (0, seg_samples - len(y_aug)), mode="constant")

      # Re-apply normalization — gain/noise augmentations change RMS energy
      y_aug = rms_normalize(y_aug)
      base = os.path.basename(f)
      save_name = f"{os.path.splitext(base)[0]}_aug-{disease}-{i}.wav"
      save_path = os.path.join(save_dir, save_name)
      sf.write(save_path, y_aug, sr)

      if not os.path.exists(save_path):
        print("Failed to save:", save_path)

      updated_records.append({
        "segment_file": os.path.join(save_name), 
        "pid": pid, 
        "disease": disease,
      })
      i += 1

  pd.DataFrame(undersampling_records).to_csv("outputs/augmentation_outputs/df_after_undersampling.csv", index=False)
  pd.DataFrame(updated_records).to_csv("outputs/augmentation_outputs/train_segments.csv")
  return pd.DataFrame(updated_records)
  
train_augmented = augment_training_set(train_segments, 
    os.path.join(ROOT_DIR, SEGMENTS_DIR, "train_segments"),
    os.path.join(ROOT_DIR, TRAIN_SEGMENTS_AUGMENTED_DIR),
)

##### Augmented Segment Distribution

In [ ]:
# Read segments data
seg_df = pd.read_csv('outputs/augmentation_outputs/train_segments.csv')

# Count segments per disease
disease_counts = seg_df['disease'].value_counts()

# Create the bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(disease_counts.index, disease_counts.values, 
               color=['skyblue', 'lightcoral', 'lightgreen', 'orange', 'purple', 'brown', 'gray'])

# Add value labels on bars
for bar, count in zip(bars, disease_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
             f'{count:,}', ha='center', va='bottom', fontweight='bold')

plt.title('Number of Audio Segments per Disease Class', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Number of Segments', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# Add some padding to top for labels
plt.ylim(0, disease_counts.max() * 1.1)

plt.tight_layout()
plt.show()

# Print the counts
print("Segment counts per disease:")
print(disease_counts)

In [ ]:
shutil.copytree(os.path.join(ROOT_DIR, SEGMENTS_DIR, "test_segments"),
                "../dataset/test-segments")

shutil.copy(os.path.join(ROOT_DIR, "outputs/segmentation_outputs/test_segments.csv"),
                "../dataset/test_segments.csv")



shutil.copytree(os.path.join(ROOT_DIR, SEGMENTS_DIR, "val_segments"),
                "../dataset/val-segments")

shutil.copy(os.path.join(ROOT_DIR, "outputs/segmentation_outputs/val_segments.csv"),
                "../dataset/val_segments.csv")



shutil.copytree(os.path.join(ROOT_DIR, TRAIN_SEGMENTS_AUGMENTED_DIR),
                "../dataset/train-segments")

shutil.copy(os.path.join(ROOT_DIR, "outputs/augmentation_outputs/train_segments.csv"),
                "../dataset/train_segments.csv")

In [ ]:
import pandas as pd
import os

# Load all three CSVs
train_df = pd.read_csv(os.path.join(ROOT_DIR, "outputs/augmentation_outputs/train_segments.csv"))
test_df  = pd.read_csv(os.path.join(ROOT_DIR, "outputs/segmentation_outputs/test_segments.csv"))
val_df   = pd.read_csv(os.path.join(ROOT_DIR, "outputs/segmentation_outputs/val_segments.csv"))

print("=" * 65)
print("PIONEER DATASET COMPOSITION REPORT")
print("=" * 65)

for split_name, df in [
    ("Train (augmented)", train_df),
    ("Test (development)", test_df),
    ("Unseen", val_df)
]:
    print(f"\n--- {split_name} ---")
    print(f"  Total segments: {len(df)}")
    counts = df.groupby("disease").agg(
        segments=("segment_file", "count"),
        patients=("pid", "nunique")
    ).reset_index()
    print(counts.to_string(index=False))

# Overall summary
all_df = pd.concat([
    train_df.assign(split="Train"),
    test_df.assign(split="Test"),
    val_df.assign(split="Unseen")
], ignore_index=True)

print(f"\n--- Overall ---")
print(f"  Total segments        : {len(all_df)}")
print(f"  Total unique patients : {all_df['pid'].nunique()}")
overall = all_df.groupby("disease").agg(
    total_segments=("segment_file", "count"),
    total_patients=("pid", "nunique")
).reset_index()
print(overall.to_string(index=False))
print("=" * 65)